# Day 38: Lab - Statistical Analysis Mini-Project

## Project Goal
A/B testing is one of the most common real-world applications of statistics in a data science role: you show two versions of something (here, an old vs. new website design) to different users and use statistics to decide whether the difference in outcomes is a real effect or just random noise. This lab simulates that end-to-end: we generate fake "control" (old design) and "treatment" (new design) conversion data, estimate each group's conversion rate with a confidence interval, run a formal hypothesis test to check if the difference is statistically significant, and finally translate that result into a plain business decision. The code below simulates 500 users per group, where each user either converts (1) or doesn't (0), using `np.random.binomial` with different underlying true conversion probabilities (10% control, 13% treatment) - in a real A/B test this data would come from actual user behavior, not simulation. The printed rates are each group's observed conversion rate from this sample.

In [ ]:
import pandas as pd
import numpy as np

# Fix the seed so this simulated experiment is reproducible
np.random.seed(42)
# Simulate 500 users per group as 0/1 outcomes (1 = converted).
# binomial(1, p, size) with n=1 trial per draw is equivalent to a weighted coin flip
control = np.random.binomial(1, 0.10, 500)      # true conversion rate 10%
treatment = np.random.binomial(1, 0.13, 500)     # true conversion rate 13%
print('Control conversion rate:', control.mean())
print('Treatment conversion rate:', treatment.mean())

## Step 1: Confidence Intervals for Each Group
Before comparing the two groups directly, it's useful to see how much uncertainty surrounds each group's own conversion rate estimate - a rate of 13% based on 500 users carries some sampling noise, and the CI quantifies that. This uses the same proportion-CI formula from Day 36: standard error `sqrt(p_hat * (1 - p_hat) / n)`, then the 95% interval is the observed rate plus or minus 1.96 standard errors. The loop below runs that calculation once for `control` and once for `treatment`, printing each group's observed rate alongside its CI. If the two groups' confidence intervals overlap substantially, that's an early visual hint the difference might not be statistically significant - though the formal test in Step 2 is what actually confirms that.

In [ ]:
from scipy import stats

# Compute a 95% CI for the conversion rate of each group independently
for name, group in [('Control', control), ('Treatment', treatment)]:
    p_hat = group.mean()
    se = (p_hat * (1 - p_hat) / len(group)) ** 0.5
    ci = (p_hat - 1.96 * se, p_hat + 1.96 * se)
    print(name, 'rate:', round(p_hat, 3), 'CI:', tuple(round(x, 3) for x in ci))

## Step 2: Hypothesis Test
Looking at two separate CIs is a rough guide, but the correct way to test "is the treatment rate really higher than control?" is a formal two-sample hypothesis test on the two groups directly. The null hypothesis is that both groups have the same true conversion rate, and the test measures how surprising the observed difference is if that null hypothesis were true. Since our data is binary (0/1) rather than continuous, the textbook-correct test is a two-proportion z-test; here we instead use `stats.ttest_ind`, an independent two-sample t-test, as a commonly used approximation - because with 500 observations per group, the 0/1 data is large enough that the t-test's assumptions hold up reasonably well (this is the CLT in action, from Day 35). The function returns a t-statistic (how many standard errors apart the two group means are) and a p-value (the probability of seeing a difference this large, or larger, if there were truly no difference between the groups).

In [ ]:
# Independent two-sample t-test comparing control vs treatment conversion (0/1 data)
t_stat, p_value = stats.ttest_ind(control, treatment)
print('t-statistic:', t_stat)
print('p-value:', p_value)

## Step 3: Conclusion
A p-value on its own doesn't tell a product manager what to do - the final, essential step of any statistical analysis is translating the result into a clear, actionable decision. Applying the standard 0.05 significance threshold from Day 37: if the p-value from the t-test is below 0.05, we reject the null hypothesis and conclude the treatment's higher conversion rate is unlikely to be due to chance alone, supporting a decision to roll out the new design. If it's 0.05 or above, we don't have strong enough evidence yet to say the new design is genuinely better - the honest recommendation is to gather more data (run the test longer) rather than conclude there's no effect, since a "not significant" result doesn't prove the effect doesn't exist, only that this sample couldn't detect it confidently.

In [ ]:
# Translate the p-value into a plain-language business decision
if p_value < 0.05:
    print('Result is statistically significant - roll out the new design')
else:
    print('Not enough evidence yet - consider running the test longer')